# Session 1: FHIR Fundamentals with LLM-Assisted Code Generation

> **INSTRUCTOR VERSION** -- This notebook contains reference implementations for all student exercises. The three code cells that are empty in the student version (Steps 1-3) are filled in here with working solutions. The clinical summary markdown cell also includes a sample response. Use this as an answer key and to demo live solutions during class.

## Clinical Scenario
**Find patients with Type 2 diabetes, retrieve their most recent HbA1c values, and identify those with poor glycemic control (HbA1c > 7.5%).**

## How This Notebook Works
- **Pre-built cells** (with code): Just run them with Shift+Enter
- **Empty cells** (with instructions above): Ask Claude to generate the code, paste it in, run it
- **Verification cells**: Run after your code to check your results
- **Markdown cells with ✏️**: Read and fill in where prompted

## What You'll Learn
- How FHIR represents clinical data (Resources, References, Bundles)
- How to query a FHIR server using Python
- How to use an LLM to generate working API calls
- Why answering clinical questions requires multi-step FHIR queries

### 📋 Clinical Code Reference

| Code | System | Meaning | Used In | ICD-10 Reference |
|------|--------|---------|---------|------------------|
| 44054006 | SNOMED CT | Type 2 Diabetes Mellitus | Condition search | E11 |
| 59621000 | SNOMED CT | Essential Hypertension | (Session 3) | I10 |
| 4548-4 | LOINC | Hemoglobin A1c (HbA1c) | Observation search | - |
| 85354-9 | LOINC | Blood Pressure panel | (Session 3) | - |
| 2160-0 | LOINC | Creatinine [Mass/volume] in Serum or Plasma | (Session 3) | - |

**HbA1c Interpretation:**
- < 5.7%: Normal
- 5.7% – 6.4%: Prediabetes
- ≥ 6.5%: Diabetes
- \> 7.5%: Poor glycemic control — needs intervention

### 🔗 How FHIR Resources Link Together

Our clinical question requires THREE types of FHIR resources:

```
Condition (code: 44054006 — Type 2 Diabetes, SNOMED CT)
  └─ subject.reference ──→ Patient/{id}
                              ↑
Observation (code: 4548-4 — HbA1c, LOINC)
  └─ subject ─────────────────┘
```

- **Condition** records a diagnosis. It points to the Patient via `subject.reference`.
- **Patient** holds demographics (name, birthdate, gender).
- **Observation** records a lab result or vital sign. It also points to the Patient.

To answer "which diabetic patients have poor HbA1c control?" we must:
1. Search Conditions → find patients with diabetes
2. Follow references → get Patient demographics
3. Search Observations → get each patient's HbA1c values
4. Combine and analyze

In [ ]:
# ============================================================
# SETUP — Run this cell first
# ============================================================
import requests
import json
import pandas as pd
from IPython.display import display, HTML

FHIR_BASE = "https://launch.smarthealthit.org/v/r4/fhir"

def show_json(data, max_lines=30):
    """Pretty-print JSON, truncated for readability."""
    text = json.dumps(data, indent=2)
    lines = text.split('\n')
    if len(lines) > max_lines:
        print('\n'.join(lines[:max_lines]))
        print(f'\n... ({len(lines) - max_lines} more lines)')
    else:
        print(text)

# Verify the server is reachable
resp = requests.get(f"{FHIR_BASE}/metadata", params={"_format": "json"}, timeout=10)
if resp.status_code == 200:
    fhir_version = resp.json().get("fhirVersion", "unknown")
    print(f"✅ Connected to FHIR server: {FHIR_BASE}")
    print(f"   FHIR version: {fhir_version}")
    print(f"   This server has synthetic (Synthea) patient data. No login required.")
else:
    print(f"❌ Could not connect. Status: {resp.status_code}. Check your internet.")

## Step 0: Your First FHIR Query

Let's start by fetching a few Patient resources to see what FHIR data looks like.

In [ ]:
# ============================================================
# YOUR FIRST FHIR QUERY — Fetching Patient resources
# ============================================================
resp = requests.get(f"{FHIR_BASE}/Patient",
    params={"_count": 3, "_format": "json"})
bundle = resp.json()

print(f"Response type: {bundle['resourceType']}")  # Always 'Bundle' for search results
print(f"Total patients on server: {bundle.get('total', 'unknown')}")
print(f"Entries returned: {len(bundle.get('entry', []))}")
print()
print("--- First Patient Resource ---")
first_patient = bundle["entry"][0]["resource"]
show_json(first_patient)

### 🔍 What Did We Just See?

The FHIR server returned a **Bundle** — a container for search results.

- `resourceType: "Bundle"` — this is a search result container
- `total` — how many resources matched on the entire server
- `entry` — an array of results, each containing a `resource`

Inside each **Patient** resource:
- `id` — the unique identifier. Other resources use this to REFERENCE this patient.
- `name` — array with `given` (first name) and `family` (last name)
- `birthDate`, `gender` — demographics

**The URL pattern:** `{server}/Patient?_count=3` means "give me up to 3 Patient resources."
You'll use this same pattern with Condition and Observation next.

## ✏️ Step 1: Find Patients with Type 2 Diabetes

Now you'll use Claude to write a FHIR query. Open the **Claude web interface** and enter this prompt:

> Write Python code using the `requests` library to search for Condition
> resources with SNOMED CT code 44054006 (Type 2 diabetes) on the FHIR server at
> https://launch.smarthealthit.org/v/r4/fhir. Limit to 20 results. For each condition
> found, extract and print:
> - The condition resource ID
> - The patient reference (from subject.reference)
> - The display name of the condition
> - The onset date (from onsetDateTime)
>
> Also collect all unique patient references into a Python set called `patient_refs`.

**Paste Claude's response in the cell below and run it (Shift+Enter).**

In [ ]:
# REFERENCE IMPLEMENTATION — Step 1: Search for Type 2 Diabetes conditions
resp = requests.get(f"{FHIR_BASE}/Condition",
    params={"code": "44054006", "_count": 20, "_format": "json"})
bundle = resp.json()

patient_refs = set()
print(f"Total conditions on server: {bundle.get('total', 'unknown')}")
print(f"Entries returned: {len(bundle.get('entry', []))}")
print()

for entry in bundle.get("entry", []):
    resource = entry["resource"]
    condition_id = resource.get("id", "")
    patient_ref = resource.get("subject", {}).get("reference", "")
    code_display = resource.get("code", {}).get("coding", [{}])[0].get("display", "")
    onset = resource.get("onsetDateTime", "unknown")
    
    if patient_ref:
        patient_refs.add(patient_ref)
    print(f"  Condition {condition_id}: {code_display}")
    print(f"    Patient: {patient_ref}, Onset: {onset}")

print(f"\nFound {len(patient_refs)} unique patients with Type 2 diabetes")

In [ ]:
# ============================================================
# VERIFICATION — Check your Condition search results
# ============================================================
try:
    print(f"✅ Found {len(patient_refs)} unique patients with Type 2 diabetes")
    print(f"   Example references: {list(patient_refs)[:5]}")
    print()
    # Extract just the IDs for the next step
    patient_ids = [ref.split('/')[-1] for ref in patient_refs if '/' in ref]
    print(f"   Extracted patient IDs: {patient_ids[:5]}")
    print(f"\n   Next: we'll FOLLOW these references to get patient demographics.")
except NameError:
    print("⚠️  Variable 'patient_refs' not found.")
    print("   Make sure your code creates a set called 'patient_refs'")
    print("   containing strings like 'Patient/abc123'")
    print()
    print("   If Claude used a different variable name, rename it and re-run,")
    print("   or uncomment the fallback below:")
    print()
    print("   # --- FALLBACK ---")
    print("   # resp = requests.get(f'{FHIR_BASE}/Condition',")
    print("   #     params={'code': '44054006', '_count': 20, '_format': 'json'})")
    print("   # bundle = resp.json()")
    print("   # patient_refs = set()")
    print("   # for entry in bundle.get('entry', []):")
    print("   #     ref = entry['resource'].get('subject', {}).get('reference', '')")
    print("   #     if ref: patient_refs.add(ref)")
    print("   # patient_ids = [ref.split('/')[-1] for ref in patient_refs]")

## ✏️ Step 2: Get Patient Demographics

Each Condition resource points to a Patient via `subject.reference` (e.g., `Patient/abc123`).
Now we follow those references to get each patient's name, birthdate, and gender.

Ask Claude:

> I have a Python list called `patient_ids` containing FHIR patient IDs like
> `["abc123", "def456"]`. Write Python code that fetches each Patient resource
> from `https://launch.smarthealthit.org/v/r4/fhir/Patient/{id}` and extracts their full name
> (combining given and family name), birth date, and gender. Store the results
> in a list of dictionaries called `patients` where each dict has keys:
> `"id"`, `"name"`, `"birthDate"`, `"gender"`. Print each patient as you fetch them.

**Paste Claude's code below.**

In [ ]:
# REFERENCE IMPLEMENTATION — Step 2: Get Patient demographics
patients = []
for pid in patient_ids:
    resp = requests.get(f"{FHIR_BASE}/Patient/{pid}",
        params={"_format": "json"}, timeout=10)
    if resp.status_code == 200:
        p = resp.json()
        name_parts = p.get("name", [{}])[0]
        full_name = f"{' '.join(name_parts.get('given', []))} {name_parts.get('family', '')}".strip()
        patient = {
            "id": p.get("id", pid),
            "name": full_name,
            "birthDate": p.get("birthDate", "unknown"),
            "gender": p.get("gender", "unknown")
        }
        patients.append(patient)
        print(f"  {full_name} (DOB: {patient['birthDate']}, {patient['gender']})")
    else:
        print(f"  \u26a0\ufe0f Could not fetch Patient/{pid}: status {resp.status_code}")

print(f"\nRetrieved {len(patients)} patient records")

In [ ]:
# ============================================================
# PATIENT DEMOGRAPHICS TABLE
# ============================================================
try:
    df_patients = pd.DataFrame(patients)
    print(f"✅ Retrieved demographics for {len(df_patients)} patients:\n")
    display(df_patients)
    print(f"\nNotice: each patient has an 'id' — we'll use this to search for their lab results.")
except NameError:
    print("⚠️  Variable 'patients' not found.")
    print("   Make sure your code creates a list called 'patients'.")
    print("   Each item: {'id': '...', 'name': '...', 'birthDate': '...', 'gender': '...'}")

## ✏️ Step 3: Retrieve HbA1c Lab Values

This is the critical step. For EACH patient, we search for Observation resources
with LOINC code **4548-4** (HbA1c). We want only the most recent result.

Ask Claude:

> I have a Python list called `patients` where each item is a dictionary with
> an `"id"` key containing a FHIR patient ID. For each patient, write Python
> code to search for Observation resources at
> `https://launch.smarthealthit.org/v/r4/fhir/Observation` with these URL parameters:
> - `subject`: `Patient/{id}`
> - `code`: `4548-4`
> - `_sort`: `-date`
> - `_count`: `1`
>
> Extract the date (`effectiveDateTime`), numeric value (`valueQuantity.value`),
> and unit (`valueQuantity.unit`) from the most recent observation.
> Store results in a list called `observations` where each dict has keys:
> `"patient_id"`, `"date"`, `"value"`, `"unit"`. If a patient has no HbA1c
> observation, include them with value `"N/A"`. Print progress.

**Paste Claude's code below.**

In [ ]:
# REFERENCE IMPLEMENTATION — Step 3: Retrieve HbA1c observations
observations = []
for patient in patients:
    pid = patient["id"]
    resp = requests.get(f"{FHIR_BASE}/Observation",
        params={
            "subject": f"Patient/{pid}",
            "code": "4548-4",
            "_sort": "-date",
            "_count": 1,
            "_format": "json"
        }, timeout=10)
    
    if resp.status_code == 200:
        bundle = resp.json()
        if bundle.get("entry"):
            obs = bundle["entry"][0]["resource"]
            value_qty = obs.get("valueQuantity", {})
            observations.append({
                "patient_id": pid,
                "date": obs.get("effectiveDateTime", "unknown"),
                "value": value_qty.get("value", "N/A"),
                "unit": value_qty.get("unit", "")
            })
            print(f"  {patient['name']}: HbA1c = {value_qty.get('value', 'N/A')} {value_qty.get('unit', '')} ({obs.get('effectiveDateTime', 'unknown')})")
        else:
            observations.append({
                "patient_id": pid,
                "date": "N/A",
                "value": "N/A",
                "unit": ""
            })
            print(f"  {patient['name']}: No HbA1c data found")

print(f"\nRetrieved observations for {len(observations)} patients")

In [ ]:
# ============================================================
# COMBINED ANALYSIS — Identify Poor Glycemic Control
# ============================================================
try:
    df_obs = pd.DataFrame(observations)
    df_patients_copy = df_patients.copy()
    df_patients_copy['id'] = df_patients_copy['id'].astype(str)
    df_obs['patient_id'] = df_obs['patient_id'].astype(str)

    df_merged = df_obs.merge(df_patients_copy, left_on='patient_id', right_on='id', how='left')

    # Convert to numeric
    df_merged['hba1c_numeric'] = pd.to_numeric(df_merged['value'], errors='coerce')

    # Flag poor control
    def control_flag(x):
        if pd.isna(x): return '⚪ No data'
        if x > 7.5: return '🔴 Poor control'
        if x >= 6.5: return '🟡 Diabetic range'
        return '🟢 Below threshold'

    df_merged['glycemic_control'] = df_merged['hba1c_numeric'].apply(control_flag)

    display_cols = [c for c in ['name','birthDate','gender','date','value','unit','glycemic_control']
                    if c in df_merged.columns]
    print('📊 Patients with Type 2 Diabetes — HbA1c Results:\n')
    display(df_merged[display_cols])

    has_data = df_merged['hba1c_numeric'].notna()
    poor = (df_merged['hba1c_numeric'] > 7.5) & has_data
    print(f'\n📈 Summary:')
    print(f'   Total diabetic patients: {len(df_merged)}')
    print(f'   With HbA1c data: {has_data.sum()}')
    print(f'   🔴 Poor control (>7.5%): {poor.sum()}')
    print(f'   🟢 Adequate: {(has_data & ~poor).sum()}')
    print(f'   ⚪ No data: {(~has_data).sum()}')

    if poor.sum() > 0:
        print(f'\n   Patients needing follow-up:')
        for _, row in df_merged[poor].iterrows():
            print(f"     • {row.get('name','?')}: HbA1c = {row['value']}%")

except NameError as e:
    print(f'⚠️  Error: {e}')
    print('   Make sure you ran Steps 1-3 successfully.')
except Exception as e:
    print(f'⚠️  Unexpected error: {e}')
    print('   Ask Claude to help debug.')

## ✏️ Step 4: Generate a Clinical Summary

We now have structured data in a table. The final step: translate this into
a narrative a clinician or patient could read.

**Copy the table output above** and paste it into the Claude web interface with:

> Here is a table of patients with Type 2 diabetes and their most recent HbA1c
> values. Write a brief clinical summary suitable for a care coordinator.
> Identify patients with poor glycemic control (HbA1c > 7.5%) and note they may
> need follow-up. Format as 2-3 short paragraphs. Use ONLY the data in the
> table — do not add any information that is not present.

**Paste Claude's summary in the markdown cell below.**

### Clinical Summary

*(Sample reference summary -- actual results will vary depending on the data returned by the SMART on FHIR sandbox server at the time of execution.)*

A review of the diabetic patient panel identified multiple patients with a documented diagnosis of Type 2 Diabetes Mellitus (SNOMED CT 44054006). For each patient, the most recent Hemoglobin A1c (HbA1c, LOINC 4548-4) laboratory value was retrieved from the electronic health record to assess glycemic control.

Of the patients reviewed, those with an HbA1c value exceeding 7.5% are flagged as having **poor glycemic control** and may benefit from clinical follow-up. This could include medication adjustment, referral to endocrinology or diabetes education, and a reassessment of dietary and lifestyle factors. Patients without available HbA1c data should be prioritized for lab work to establish a current baseline.

All patients in this cohort should have their care plans reviewed at the next scheduled visit. For those with elevated HbA1c, a follow-up lab draw in 3 months is recommended to assess response to any interventions. This summary is generated from structured FHIR data and should be validated by the care team before clinical action is taken.

## 🧠 Session 1 Reflection

You just manually executed a **three-step clinical data pipeline**:

1. **Condition search** → Found patients with Type 2 diabetes (SNOMED CT: 44054006)
2. **Patient lookup** → Retrieved demographics by following `subject.reference`
3. **Observation search** → Got HbA1c lab values (LOINC: 4548-4) for each patient

You used an LLM (Claude) in two distinct roles:
- **Code generation** — Claude wrote the Python/FHIR queries
- **Summarization** — Claude translated structured data into clinical narrative

**Key insight:** The LLM never touched the FHIR server directly. YOUR CODE did
the querying. The LLM helped you *write* the code and *interpret* the results.
This separation between planning/interpretation (LLM) and execution (code) is
critical for safety and correctness in clinical systems.

**Next session:** What if the LLM could orchestrate these same steps
*autonomously* — deciding which queries to run and in what order? That's called
**tool use**, and it's the foundation of AI agents.

### 💾 Save this notebook — you'll reference it in Session 2.